In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)  # show all columns in DataFrames
%matplotlib inline

In [ ]:
# Download daily GSPC data — start from 2010 to give the 200-day MA enough
# warmup history before our target period begins in 2020.
# Without this buffer, ma_200 would be NaN for the first ~200 trading days
# and dropna() would eliminate all of 2020 from the dataset.
df = yf.download("^GSPC", start="2010-01-01", end="2025-12-31", interval="1d")

# Flatten multi-level columns if present (yfinance sometimes returns them)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Basic sanity checks
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head(10)

In [ ]:
# Plot SPY closing price over the full period
plt.figure(figsize=(14, 5))
plt.plot(df["Close"], linewidth=1)
plt.title("SPY Daily Closing Price (2020–2025)")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.grid(True)
plt.show()

In [ ]:
# Summary statistics for all OHLCV columns
df.describe()

In [ ]:
# Work on a copy to keep the original raw data untouched
features = df.copy()

# ── PRICE-BASED FEATURES ──────────────────────────────────────────────────────

features['daily_return']    = features['Close'].pct_change()
features['high_low_range']  = (features['High'] - features['Low']) / features['Close']
features['open_close_range']= (features['Close'] - features['Open']) / features['Open']

# Short-term moving averages — capture near-term trend direction
features['ma_5']  = features['Close'].rolling(5).mean()
features['ma_10'] = features['Close'].rolling(10).mean()
features['ma_20'] = features['Close'].rolling(20).mean()

# Long-term moving averages — institutional-grade trend signals
# 50-day MA: medium-term trend; 200-day MA: long-term bull/bear regime
features['ma_50']  = features['Close'].rolling(50).mean()
features['ma_200'] = features['Close'].rolling(200).mean()

# Price relative to moving average (> 1.0 = above MA = bullish, < 1.0 = bearish)
features['close_to_ma5']   = features['Close'] / features['ma_5']
features['close_to_ma20']  = features['Close'] / features['ma_20']
features['close_to_ma50']  = features['Close'] / features['ma_50']
features['close_to_ma200'] = features['Close'] / features['ma_200']

# Golden/death cross signal: 50-day vs 200-day MA ratio
# > 1.0 → golden cross (bullish regime); < 1.0 → death cross (bearish regime)
features['ma50_200_ratio'] = features['ma_50'] / features['ma_200']

# Medium-term momentum
features['momentum_5']  = features['Close'].pct_change(5)
features['momentum_10'] = features['Close'].pct_change(10)

# Rolling volatility — higher std = more uncertain environment
features['volatility_5']  = features['daily_return'].rolling(5).std()
features['volatility_10'] = features['daily_return'].rolling(10).std()
features['volatility_20'] = features['daily_return'].rolling(20).std()

# Bollinger Band position — how stretched is price relative to recent range?
rolling_std = features['Close'].rolling(20).std()
bb_upper = features['ma_20'] + 2 * rolling_std
bb_lower = features['ma_20'] - 2 * rolling_std
features['bb_position'] = (features['Close'] - bb_lower) / (bb_upper - bb_lower)

# Average True Range — a cleaner volatility measure than rolling std of returns
high_low   = features['High'] - features['Low']
high_close = (features['High'] - features['Close'].shift(1)).abs()
low_close  = (features['Low']  - features['Close'].shift(1)).abs()
features['atr_14'] = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1).rolling(14).mean()

# Week-of-year — SPY has mild seasonality (Jan effect, Sep weakness)
features['week_of_year'] = features.index.isocalendar().week.astype(float)

# ── VOLUME FEATURES ───────────────────────────────────────────────────────────

features['volume_ma_5']  = features['Volume'].rolling(5).mean()
# Volume ratio > 1 = unusual activity, often precedes large moves
features['volume_ratio'] = features['Volume'] / features['volume_ma_5']

# ── RSI ───────────────────────────────────────────────────────────────────────
# RSI > 70 → overbought; RSI < 30 → oversold
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.where(delta > 0, 0).ewm(alpha=1/period, adjust=False).mean()  # Wilder's smoothing
    loss  = -delta.where(delta < 0, 0).ewm(alpha=1/period, adjust=False).mean()
    rs    = gain / loss
    return 100 - (100 / (1 + rs))

features['rsi_14'] = compute_rsi(features['Close'])

# ── MACD ──────────────────────────────────────────────────────────────────────
ema12 = features['Close'].ewm(span=12, adjust=False).mean()
ema26 = features['Close'].ewm(span=26, adjust=False).mean()
features['macd']        = ema12 - ema26
features['macd_signal'] = features['macd'].ewm(span=9, adjust=False).mean()
features['macd_hist']   = features['macd'] - features['macd_signal']

# Drop rows where rolling windows didn't have enough history
# Note: ma_200 requires 200 days, so the first ~200 rows are dropped
features.dropna(inplace=True)

print(f"Shape after feature engineering: {features.shape}")
print(f"Date range after dropna: {features.index.min().date()} → {features.index.max().date()}")
print(f"\nFeature columns ({len(features.columns)}):\n{features.columns.tolist()}")

In [ ]:
# Define which columns are features (everything except raw OHLCV)
feature_cols = [
    'daily_return', 'high_low_range', 'open_close_range',
    'ma_5', 'ma_10', 'ma_20', 'ma_50', 'ma_200',
    'close_to_ma5', 'close_to_ma20', 'close_to_ma50', 'close_to_ma200',
    'ma50_200_ratio',
    'momentum_5', 'momentum_10',
    'volatility_5', 'volatility_10', 'volatility_20',
    'volume_ma_5', 'volume_ratio',
    'rsi_14', 'macd', 'macd_signal', 'macd_hist'
]

weekly = pd.DataFrame()

# ── AGGREGATE OHLC ────────────────────────────────────────────────────────────
weekly_open  = features['Open'].resample('W').first()   # first open of the week
weekly_high  = features['High'].resample('W').max()     # highest high of the week
weekly_low   = features['Low'].resample('W').min()      # lowest low of the week
weekly_close = features['Close'].resample('W').last()   # last close of the week

# ── LEAKAGE-FREE FEATURE AGGREGATION ─────────────────────────────────────────
# Step 1: aggregate daily features to weekly means (this week's values)
weekly_features_current = features[feature_cols].resample('W').mean()

# Step 2: shift features forward by 1 week so that row[t] contains last week's
# aggregated values. This guarantees the model only sees completed prior-week
# data when predicting the current week's OHLC — no mid-week leakage.
weekly[feature_cols] = weekly_features_current.shift(1)

# ── LAG FEATURES ─────────────────────────────────────────────────────────────
# Key indicators to lag — these capture the most useful recent-history signals
lag_cols = ['rsi_14', 'macd', 'macd_hist', 'momentum_5', 'volatility_5',
            'close_to_ma20', 'close_to_ma200', 'volume_ratio', 'daily_return']

for col in lag_cols:
    for lag in [1, 2, 3]:
        # shift(lag) on already-shifted weekly features gives us lag+1 weeks back
        # e.g. lag=1 → 2 weeks ago, lag=2 → 3 weeks ago, etc.
        weekly[f'{col}_lag{lag}'] = weekly_features_current[col].shift(1 + lag)

# Store current week's close — used as the reference point for % change targets
weekly['current_close'] = weekly_close

# ── TARGETS ───────────────────────────────────────────────────────────────────
# Targets: next week's price expressed as % change from current week's close
weekly['target_open']  = (weekly_open.shift(-1)  - weekly_close) / weekly_close
weekly['target_high']  = (weekly_high.shift(-1)  - weekly_close) / weekly_close
weekly['target_low']   = (weekly_low.shift(-1)   - weekly_close) / weekly_close
weekly['target_close'] = (weekly_close.shift(-1) - weekly_close) / weekly_close

# Drop rows with NaN — introduced by the shift(1) on features (first row)
# and shift(-1) on targets (last row)
weekly.dropna(inplace=True)

lag_feature_count = len(lag_cols) * 3
print(f"Weekly dataset shape: {weekly.shape}")
print(f"Original features:    {len(feature_cols)}")
print(f"Lag features added:   {lag_feature_count}  ({len(lag_cols)} indicators × 3 lags)")
print(f"Total feature count:  {len(feature_cols) + lag_feature_count}")
print(f"\nSample targets (% change from current close):")
print(weekly[['target_open', 'target_high', 'target_low', 'target_close']].head())

In [ ]:
# Separate features from targets
feature_cols_only = [col for col in feature_cols if col in weekly.columns]
X = weekly[feature_cols_only]
y = weekly[['target_open', 'target_high', 'target_low', 'target_close']]

# Keep current_close aside — needed later to convert % predictions back to prices
current_close = weekly['current_close']

# Compute chronological split indices
total     = len(weekly)
train_end = int(total * 0.70)
val_end   = int(total * 0.85)

print(f"Total weeks:      {total}")
print(f"Training weeks:   {train_end}")
print(f"Validation weeks: {val_end - train_end}")
print(f"Test weeks:       {total - val_end}")

# Slice all arrays at the same boundaries
X_train, X_val, X_test = X.iloc[:train_end], X.iloc[train_end:val_end], X.iloc[val_end:]
y_train, y_val, y_test = y.iloc[:train_end], y.iloc[train_end:val_end], y.iloc[val_end:]

cc_train = current_close.iloc[:train_end]
cc_val   = current_close.iloc[train_end:val_end]
cc_test  = current_close.iloc[val_end:]

print(f"\nDate ranges:")
print(f"Train:      {X_train.index.min().date()} → {X_train.index.max().date()}")
print(f"Validation: {X_val.index.min().date()}   → {X_val.index.max().date()}")
print(f"Test:       {X_test.index.min().date()}   → {X_test.index.max().date()}")

In [ ]:
# Visualise the split — confirms no overlap between sets
plt.figure(figsize=(14, 4))
plt.plot(X_train.index, y_train['target_close'], label='Train',      color='steelblue')
plt.plot(X_val.index,   y_val['target_close'],   label='Validation', color='orange')
plt.plot(X_test.index,  y_test['target_close'],  label='Test',       color='green')
plt.title('70 / 15 / 15 Train-Validation-Test Split (Weekly target_close)')
plt.xlabel('Date')
plt.ylabel('Weekly % Change in Close')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

models = {}

for target in ['target_open', 'target_high', 'target_low', 'target_close']:
    rf = RandomForestRegressor(
        n_estimators=200,     # number of trees in the forest
        max_depth=5,          # max tree depth — shallower trees generalise better
        min_samples_leaf=5,   # each leaf must have at least 5 samples
        random_state=42,      # reproducibility
        n_jobs=-1             # parallelise across all CPU cores
    )
    rf.fit(X_train, y_train[target])
    models[target] = rf
    print(f"✓ Trained model for {target}")

print("\nAll models trained!")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=== BASELINE VALIDATION SET PERFORMANCE ===\n")

val_predictions = {}

for target in ['target_open', 'target_high', 'target_low', 'target_close']:
    preds = models[target].predict(X_val)
    val_predictions[target] = preds

    mae  = mean_absolute_error(y_val[target], preds)
    rmse = np.sqrt(mean_squared_error(y_val[target], preds))
    r2   = r2_score(y_val[target], preds)

    print(f"{target.replace('target_', '').upper()}")
    print(f"  MAE:  {mae:.4f}  ({mae*100:.2f}%)")
    print(f"  RMSE: {rmse:.4f} ({rmse*100:.2f}%)")
    print(f"  R²:   {r2:.4f}")
    print()

In [ ]:
# Plot baseline predictions vs actuals on the validation set
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, target in enumerate(['target_open', 'target_high', 'target_low', 'target_close']):
    # Convert % change back to price: price = (1 + pct_change) * reference_close
    actual_prices    = (1 + y_val[target]) * cc_val
    predicted_prices = (1 + val_predictions[target]) * cc_val.values

    axes[i].plot(y_val.index, actual_prices.values,
                 label='Actual', color='steelblue', linewidth=1.5)
    axes[i].plot(y_val.index, predicted_prices,
                 label='Predicted', color='orange', linewidth=1.5, linestyle='--')
    axes[i].set_title(target.replace('target_', '').upper())
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('Price (USD)')
    axes[i].legend()
    axes[i].grid(True)

plt.suptitle('Random Forest — Baseline Validation Set: Predicted vs Actual', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from sklearn.metrics import r2_score

# Tightened parameter grid — max_depth capped at 7, min_samples_leaf starts at 5
param_grid = {
    'n_estimators':      [100, 200, 500],
    'max_depth':         [3, 5, 7],        # removed 10 — too deep for this dataset
    'min_samples_leaf':  [5, 10, 20],      # raised minimum — more conservative
    'min_samples_split': [10, 20],         # new: minimum samples to split a node
}

# TimeSeriesSplit: 5 folds, each fold's training set expands chronologically
tscv = TimeSeriesSplit(n_splits=5)

best_params = {}
best_scores = {}

for target in ['target_open', 'target_high', 'target_low', 'target_close']:
    best_cv_r2 = -999
    best_param  = None

    for params in ParameterGrid(param_grid):
        fold_scores = []

        for train_idx, val_idx in tscv.split(X_train):
            X_fold_train = X_train.iloc[train_idx]
            y_fold_train = y_train.iloc[train_idx][target]
            X_fold_val   = X_train.iloc[val_idx]
            y_fold_val   = y_train.iloc[val_idx][target]

            rf = RandomForestRegressor(
                n_estimators     = params['n_estimators'],
                max_depth        = params['max_depth'],
                min_samples_leaf = params['min_samples_leaf'],
                min_samples_split= params['min_samples_split'],
                max_features     = 'sqrt',   # each tree sees sqrt(n_features) — reduces overfitting
                random_state     = 42,
                n_jobs           = -1
            )
            rf.fit(X_fold_train, y_fold_train)
            fold_scores.append(r2_score(y_fold_val, rf.predict(X_fold_val)))

        # Average R² across all 5 folds — much more robust than single-window score
        mean_cv_r2 = np.mean(fold_scores)

        if mean_cv_r2 > best_cv_r2:
            best_cv_r2 = mean_cv_r2
            best_param  = params

    best_params[target] = best_param
    best_scores[target] = best_cv_r2

    print(f"{target.replace('target_', '').upper()}")
    print(f"  Best mean CV R²: {best_cv_r2:.4f}")
    print(f"  Best params:     {best_param}\n")

In [ ]:
tuned_models = {}

for target in ['target_open', 'target_high', 'target_low', 'target_close']:
    params = best_params[target]

    rf = RandomForestRegressor(
        n_estimators      = params['n_estimators'],
        max_depth         = params['max_depth'],
        min_samples_leaf  = params['min_samples_leaf'],
        min_samples_split = params['min_samples_split'],
        max_features      = 'sqrt',   # consistent with tuning
        oob_score         = True,     # free generalisation estimate using unused bootstrap samples
        random_state      = 42,
        n_jobs            = -1
    )

    # Retrain on train + validation combined for maximum data before final test
    X_train_val = pd.concat([X_train, X_val])
    y_train_val = pd.concat([y_train, y_val])

    rf.fit(X_train_val, y_train_val[target])
    tuned_models[target] = rf

    # OOB score: R² estimated on out-of-bag samples — a free sanity check
    # If OOB score ≈ validation score, the model is not badly overfit
    print(f"✓ {target.replace('target_', '').upper():6s}  OOB R²: {rf.oob_score_:.4f}")

In [ ]:
print("=== TUNED MODEL — VALIDATION PERFORMANCE ===\n")

tuned_val_predictions = {}

for target in ['target_open', 'target_high', 'target_low', 'target_close']:
    # Retrain on training data only so validation remains genuinely unseen
    rf = RandomForestRegressor(**best_params[target], random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train[target])
    preds = rf.predict(X_val)
    tuned_val_predictions[target] = preds

    mae  = mean_absolute_error(y_val[target], preds)
    rmse = np.sqrt(mean_squared_error(y_val[target], preds))
    r2   = r2_score(y_val[target], preds)

    print(f"{target.replace('target_', '').upper()}")
    print(f"  MAE:  {mae:.4f}  ({mae*100:.2f}%)")
    print(f"  RMSE: {rmse:.4f} ({rmse*100:.2f}%)")
    print(f"  R²:   {r2:.4f}")
    print()

In [ ]:
# Plot feature importance for each target using the tuned hyperparameters
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

importance_dfs = {}

for i, target in enumerate(['target_open', 'target_high', 'target_low', 'target_close']):
    # Retrain on training data only for clean importance scores
    rf = RandomForestRegressor(**best_params[target], random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train[target])

    importances = pd.Series(rf.feature_importances_, index=X_train.columns)
    importances = importances.sort_values(ascending=True)
    importance_dfs[target] = importances

    importances.plot(kind='barh', ax=axes[i], color='steelblue')
    axes[i].set_title(f"{target.replace('target_', '').upper()} — Feature Importance")
    axes[i].set_xlabel("Importance Score")
    axes[i].axvline(x=0.02, color='red', linestyle='--', label='0.02 threshold')
    axes[i].legend()

plt.suptitle('Feature Importance per Target (tuned models)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Keep features that exceed the 2% threshold for at least one target
threshold = 0.02
strong_features = set()

for target in ['target_open', 'target_high', 'target_low', 'target_close']:
    important = importance_dfs[target][importance_dfs[target] >= threshold].index.tolist()
    strong_features.update(important)

strong_features = list(strong_features)

print(f"Original feature count:  {X_train.shape[1]}")
print(f"Features kept:           {len(strong_features)}")
print(f"Features dropped:        {X_train.shape[1] - len(strong_features)}")
print(f"\nKept features:\n{sorted(strong_features)}")

# Rebuild feature matrices using only the selected features
X_train_sel = X_train[strong_features]
X_val_sel   = X_val[strong_features]
X_test_sel  = X_test[strong_features]

In [ ]:
step_size = 4  # retrain every 4 weeks

# Combine train + validation for the walk-forward pool
X_wf  = pd.concat([X_train_sel, X_val_sel])
y_wf  = pd.concat([y_train, y_val])
cc_wf = pd.concat([cc_train, cc_val])

min_train_size = len(X_train_sel)  # don't predict until we have enough history

wf_predictions = {t: [] for t in ['target_open', 'target_high', 'target_low', 'target_close']}
wf_actuals     = {t: [] for t in ['target_open', 'target_high', 'target_low', 'target_close']}
wf_dates       = []

for start in range(min_train_size, len(X_wf), step_size):
    end = min(start + step_size, len(X_wf))

    # Expanding training window: all data up to current position
    X_window_train = X_wf.iloc[:start]
    y_window_train = y_wf.iloc[:start]

    # Prediction window: next step_size weeks
    X_window_test = X_wf.iloc[start:end]
    y_window_test = y_wf.iloc[start:end]

    wf_dates.extend(X_wf.index[start:end])

    for target in ['target_open', 'target_high', 'target_low', 'target_close']:
        rf = RandomForestRegressor(**best_params[target], random_state=42, n_jobs=-1)
        rf.fit(X_window_train, y_window_train[target])
        preds = rf.predict(X_window_test)
        wf_predictions[target].extend(preds)
        wf_actuals[target].extend(y_window_test[target].values)

print("=== WALK-FORWARD VALIDATION PERFORMANCE ===\n")

for target in ['target_open', 'target_high', 'target_low', 'target_close']:
    actuals = np.array(wf_actuals[target])
    preds   = np.array(wf_predictions[target])

    mae  = mean_absolute_error(actuals, preds)
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    r2   = r2_score(actuals, preds)

    print(f"{target.replace('target_', '').upper()}")
    print(f"  MAE:  {mae:.4f}  ({mae*100:.2f}%)")
    print(f"  RMSE: {rmse:.4f} ({rmse*100:.2f}%)")
    print(f"  R²:   {r2:.4f}")
    print()

In [ ]:
# Visualise walk-forward predictions vs actuals
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
wf_dates_idx = pd.DatetimeIndex(wf_dates)

for i, target in enumerate(['target_open', 'target_high', 'target_low', 'target_close']):
    actuals = np.array(wf_actuals[target])
    preds   = np.array(wf_predictions[target])

    cc_wf_aligned    = cc_wf.reindex(wf_dates_idx)
    actual_prices    = (1 + actuals) * cc_wf_aligned.values
    predicted_prices = (1 + preds)   * cc_wf_aligned.values

    axes[i].plot(wf_dates_idx, actual_prices,
                 label='Actual', color='steelblue', linewidth=1.5)
    axes[i].plot(wf_dates_idx, predicted_prices,
                 label='Predicted', color='orange', linewidth=1.5, linestyle='--')
    axes[i].set_title(f"{target.replace('target_', '').upper()} — Walk-Forward")
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('Price (USD)')
    axes[i].legend()
    axes[i].grid(True)

plt.suptitle('Random Forest — Walk-Forward Validation: Predicted vs Actual', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print("=== FINAL TEST SET PERFORMANCE ===\n")

test_predictions = {}

for target in ['target_open', 'target_high', 'target_low', 'target_close']:
    # Use tuned models trained on train + val combined
    preds = tuned_models[target].predict(X_test)
    test_predictions[target] = preds

    mae  = mean_absolute_error(y_test[target], preds)
    rmse = np.sqrt(mean_squared_error(y_test[target], preds))
    r2   = r2_score(y_test[target], preds)

    print(f"{target.replace('target_', '').upper()}")
    print(f"  MAE:  {mae:.4f}  ({mae*100:.2f}%)")
    print(f"  RMSE: {rmse:.4f} ({rmse*100:.2f}%)")
    print(f"  R²:   {r2:.4f}")
    print()

In [ ]:
# Plot final test predictions vs actuals
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, target in enumerate(['target_open', 'target_high', 'target_low', 'target_close']):
    actual_prices    = (1 + y_test[target].values) * cc_test.values
    predicted_prices = (1 + test_predictions[target]) * cc_test.values

    axes[i].plot(y_test.index, actual_prices,
                 label='Actual', color='steelblue', linewidth=1.5)
    axes[i].plot(y_test.index, predicted_prices,
                 label='Predicted', color='orange', linewidth=1.5, linestyle='--')
    axes[i].set_title(f"{target.replace('target_', '').upper()} — Test Set")
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('Price (USD)')
    axes[i].legend()
    axes[i].grid(True)

plt.suptitle('Random Forest — Final Test Set: Predicted vs Actual', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table comparing validation vs test metrics across all targets
summary_data = []

for target in ['target_open', 'target_high', 'target_low', 'target_close']:
    val_preds  = tuned_val_predictions[target]
    val_mae    = mean_absolute_error(y_val[target], val_preds)
    val_r2     = r2_score(y_val[target], val_preds)

    test_preds = test_predictions[target]
    test_mae   = mean_absolute_error(y_test[target], test_preds)
    test_r2    = r2_score(y_test[target], test_preds)

    summary_data.append({
        'Target'   : target.replace('target_', '').upper(),
        'Val MAE'  : f"{val_mae*100:.2f}%",
        'Val R²'   : f"{val_r2:.4f}",
        'Test MAE' : f"{test_mae*100:.2f}%",
        'Test R²'  : f"{test_r2:.4f}",
    })

summary_df = pd.DataFrame(summary_data).set_index('Target')
print("=== VALIDATION vs TEST SUMMARY ===\n")
print(summary_df.to_string())